In [628]:
pip install pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [629]:
import pandas as pd
import matplotlib.pyplot as plt

SUSPICIOUS_LIMIT = 10000.00


In [630]:
file_url = "../../work/documents/transactions.csv"

try:
    df = pd.read_csv(file_url)

except FileNotFoundError:
    print(f"Erro: arquivo '{file_url}' não encontrado.")
    exit()

In [631]:
df["id"] = pd.to_numeric(df["id"], errors="coerce")
df["valor"] = pd.to_numeric(df["valor"], errors="coerce")

df["data"] = pd.to_datetime(
    df["data"],
    format="%Y-%m-%d",
    errors="coerce"
)

mask_valid = (
    df["id"].notna()
    & df["cliente_id"].notna()
    & (df["cliente_id"].astype(str).str.strip() != "")
    & df["data"].notna()
    & df["tipo"].isin(["credito", "debito"])
    & df["valor"].notna()
    & (df["valor"] > 0)
)

total_read = len(df)

df_valid = df[mask_valid].copy()
df_invalid = df[~mask_valid].copy()

print(f"Total de linhas lidas: {total_read}")
print(f"Linhas válidas: {len(df_valid)}")
print(f"Linhas inválidas: {len(df_invalid)}")

Total de linhas lidas: 20
Linhas válidas: 15
Linhas inválidas: 5


In [632]:
df_valid["mes"] = df_valid["data"].dt.strftime("%Y-%m")

monthly_report = {}

for month, group in df_valid.groupby("mes"):

    total_credito = group.loc[
        group["tipo"] == "credito",
        "valor"
    ].sum()

    total_debito = group.loc[
        group["tipo"] == "debito",
        "valor"
    ].sum()

    monthly_report[month] = {
        "quantidade": int(len(group)),
        "total_credito": round(float(total_credito), 2),
        "total_debito": round(float(total_debito), 2),
        "saldo": round(float(total_credito - total_debito), 2),
        "media": round(float(group["valor"].mean()), 2),
        "maior_valor": round(float(group["valor"].max()), 2),
        "menor_valor": round(float(group["valor"].min()), 2)
    }

In [633]:
suspects = df_valid[
    df_valid["valor"] > SUSPICIOUS_LIMIT
]

print("\n===== TRANSAÇÕES SUSPEITAS =====")

if suspects.empty:
    print("Nenhuma transação suspeita encontrada.")
else:
    for _, t in suspects.iterrows():
        valor = (
            f"{t['valor']:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

        print(
            f"ID: {int(t['id'])} | "
            f"Cliente: {t['cliente_id']} | "
            f"Data: {t['data'].strftime('%Y-%m-%d')} | "
            f"Valor: R$ {valor}"
        )


===== TRANSAÇÕES SUSPEITAS =====
ID: 5 | Cliente: CLI003 | Data: 2026-02-14 | Valor: R$ 15.000,00
ID: 10 | Cliente: CLI002 | Data: 2026-03-28 | Valor: R$ 12.000,00


In [634]:
print("\n===== RESUMO MENSAL =====")

for mes, dados in monthly_report.items():
    print(f"\nMês: {mes}")
    for chave, valor in dados.items():
        print(f"{chave}: {valor}")


===== RESUMO MENSAL =====

Mês: 2026-01
quantidade: 3
total_credito: 3500.0
total_debito: 430.5
saldo: 3069.5
media: 1310.17
maior_valor: 3500.0
menor_valor: 180.5

Mês: 2026-02
quantidade: 3
total_credito: 15000.0
total_debito: 740.9
saldo: 14259.1
media: 5246.97
maior_valor: 15000.0
menor_valor: 320.0

Mês: 2026-03
quantidade: 4
total_credito: 15500.0
total_debito: 879.9
saldo: 14620.1
media: 4094.97
maior_valor: 12000.0
menor_valor: 99.9

Mês: 2026-04
quantidade: 3
total_credito: 4300.0
total_debito: 770.45
saldo: 3529.55
media: 1690.15
maior_valor: 4300.0
menor_valor: 210.45

Mês: 2026-05
quantidade: 2
total_credito: 2750.0
total_debito: 89.99
saldo: 2660.01
media: 1419.99
maior_valor: 2750.0
menor_valor: 89.99


In [635]:
months = sorted(monthly_report.keys())
balances = [monthly_report[mes]["saldo"] for mes in months]

plt.figure(figsize=(8, 5))
plt.bar(months, balances)

plt.title("Saldo Mensal das Transações")
plt.xlabel("Mês")
plt.ylabel("Saldo (Crédito - Débito)")

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()

print("Gráfico salvo como grafico.png")

Gráfico salvo como grafico.png


In [636]:
debitos = [monthly_report[month]["total_debito"] for month in months]

plt.figure(figsize=(8, 5))
plt.plot(months, debitos, marker="o", label="Débitos")

plt.title("Evolução dos Débitos por Mês")
plt.xlabel("Mês")
plt.ylabel("Valor (R$)")
plt.legend()

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()

In [637]:
months = sorted(monthly_report.keys())

credits = [monthly_report[mes]["total_credito"] for mes in months]
debits = [monthly_report[mes]["total_debito"] for mes in months]

plt.bar(months, credits, label="Crédito")
plt.bar(months, debits, bottom=credits, label="Débito")

plt.title("Créditos e Débitos por Mês")
plt.xlabel("Mês")
plt.ylabel("Valor (R$)")
plt.legend()

plt.tight_layout()
plt.savefig("grafico.png")
plt.close()